# Milestone 3: Analisis Temporal dan Ruang + Visualisasi Interaktif

Lanjutkan dari hasil `02_data_cleaning.ipynb`. Notebook ini menganalisis pola **temporal**
(tahun, bulan, hari, jam) dan **ruang halaman** (namespace, halaman, kontributor) pada dataset
revisi Wikipedia Bahasa Indonesia (`idwiki_revisions_clean.parquet`, 27.771.250 baris x 20 kolom)
lalu menyajikannya dalam **8 visualisasi interaktif Plotly** serta insight analitik berangka.

| Item | Nilai |
|---|---|
| Milestone | **Milestone 3** (target check-in Pertemuan 8) |
| Deliverable | Analisis temporal/ruang, minimal 6 visualisasi interaktif (Plotly), insight analitik |
| Rubrik | Visualisasi dan insight (25%) + konsistensi peran Polars/DuckDB |

## Pertanyaan analisis

1. Bagaimana tren jumlah revisi per tahun dan bagaimana perbandingan penyunting anonim vs terdaftar?
2. Bagaimana pola musiman (tahun x bulan) dan bulan mana yang menonjol?
3. Kapan jam puncak aktivitas penyuntingan (UTC/WIB) dan seberapa kuat pola hari dalam seminggu?
4. Seberapa besar `page_namespace` (ruang halaman) yang paling aktif?
5. Halaman dan kontributor mana yang paling banyak berkontribusi, dan seberapa terkonsentrasi revisinya?
6. Apakah ukuran teks artikel (`text_bytes`) tumbuh dari tahun ke tahun?

## Prinsip eksekusi (penting)

- **DuckDB** menjalankan seluruh *scan* dan agregasi berat pada 27,7 juta baris
  (`memory_limit` 1 GB + *spill* ke `/tmp`), konsisten dengan pembagian peran di kuliah.
- **Polars** dipakai untuk transformasi turunan kecil pada hasil agregat
  (rolling mean, pivot matriks, rasio, YoY) -- hemat memori karena tidak menyentuh data mentah.
- Seluruh angka dihitung ulang dari data saat notebook dijalankan; **tidak ada angka hard-coded**.
- Semua timestamp disimpan dalam **UTC**; konversi ke WIB = UTC + 7.


In [1]:
from pathlib import Path
import time

import duckdb
import polars as pl
import plotly
import plotly.express as px
import plotly.graph_objects as go

CLEAN_PATH = Path('data/processed/idwiki_revisions_clean.parquet')
if not CLEAN_PATH.exists():  # fallback bila kernel berjalan dari folder notebooks/
    CLEAN_PATH = Path('..') / CLEAN_PATH
assert CLEAN_PATH.exists(), f'File tidak ditemukan: {CLEAN_PATH.resolve()}'

con = duckdb.connect()                 # in-memory DuckDB
con.execute("SET memory_limit='1GB'")  # aman untuk container ~4 GB
con.execute("SET temp_directory='/tmp'")  # spill ke disk, bukan ke repo
con.execute("SET preserve_insertion_order=false")  # hemat memori pada agregasi hash besar
con.execute(f"CREATE OR REPLACE VIEW revisi AS SELECT * FROM read_parquet('{CLEAN_PATH.as_posix()}')")

# BASE = root repo bila kernel berjalan di root, atau '..' bila berjalan di folder notebooks/
BASE = Path('.') if Path('data').exists() else Path('..')
FIG_DIR = BASE / 'output' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)


def sql(query: str) -> pl.DataFrame:
    """Jalankan agregasi DuckDB, catat waktunya, kembalikan DataFrame Polars."""
    t0 = time.perf_counter()
    df = con.execute(query).pl()
    pertama = query.strip().splitlines()[0][:66]
    print(f'{time.perf_counter() - t0:5.1f}s  {pertama}')
    return df


print('CLEAN_PATH :', CLEAN_PATH)
print('ukuran     :', f'{CLEAN_PATH.stat().st_size / 1e9:.2f} GB')
print('polars     :', pl.__version__)
print('duckdb     :', duckdb.__version__)
print('plotly     :', plotly.__version__)
print('figur      :', FIG_DIR)


CLEAN_PATH : ../data/processed/idwiki_revisions_clean.parquet
ukuran     : 1.79 GB
polars     : 2.0.0
duckdb     : 1.5.6
plotly     : 7.1.0
figur      : ../output/figures


## 1. Agregasi temporal (DuckDB)

Empat agregasi dasar yang menjadi sumber seluruh figur temporal:

1. **Per tahun** -- jumlah revisi, halaman aktif, kontributor unik, serta rasio anonim/minor/komentar.
2. **Per bulan** -- deret waktu 2003-01 sampai 2026-09 untuk melihat musiman dan lonjakan.
3. **Hari x jam (UTC)** -- 7 x 24 sel untuk pola aktivitas harian.
4. **Tahun x bulan** -- matriks musiman (kalender revisi).

> Catatan: data berakhir **1 September 2026**, sehingga tahun 2026 hanya 8 bulan --
> angka YoY untuk 2026 tidak sebanding dengan tahun penuh lainnya.


In [2]:
per_tahun = sql('''
    SELECT
        year(rev_timestamp) AS tahun,
        count(*) AS jumlah_revisi,
        count(DISTINCT page_id) AS halaman_aktif,
        count(DISTINCT contributor_name) AS kontributor_unik,
        sum(CASE WHEN is_anonymous THEN 1 ELSE 0 END)::BIGINT AS revisi_anonim,
        sum(CASE WHEN is_minor THEN 1 ELSE 0 END)::BIGINT AS revisi_minor,
        sum(CASE WHEN has_comment THEN 1 ELSE 0 END)::BIGINT AS revisi_berkomentar
    FROM revisi
    GROUP BY 1
    ORDER BY 1
''').with_columns(
    (pl.col('jumlah_revisi') / pl.col('jumlah_revisi').sum() * 100).round(2).alias('pct_total'),
    (pl.col('revisi_anonim') / pl.col('jumlah_revisi') * 100).round(1).alias('pct_anonim'),
    (pl.col('revisi_minor') / pl.col('jumlah_revisi') * 100).round(1).alias('pct_minor'),
    ((pl.col('jumlah_revisi') / pl.col('jumlah_revisi').shift(1) - 1) * 100).round(1).alias('yoy_pct'),
)

print(f'{len(per_tahun)} tahun tercakup')
print(f"Total: {per_tahun['jumlah_revisi'].sum():,} revisi")
per_tahun.tail(8)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  5.5s  SELECT
24 tahun tercakup
Total: 27,771,250 revisi


tahun,jumlah_revisi,halaman_aktif,kontributor_unik,revisi_anonim,revisi_minor,revisi_berkomentar,pct_total,pct_anonim,pct_minor,yoy_pct
i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64
2019,1682470,546918,69699,103743,964136,1425724,6.06,6.2,57.3,58.5
2020,1251689,417596,77017,120375,441652,891035,4.51,9.6,35.3,-25.6
2021,1738178,617530,76036,181620,498413,1297721,6.26,10.4,28.7,38.9
2022,2641295,820963,89545,244515,1113042,2129831,9.51,9.3,42.1,52.0
2023,2438911,957216,90238,255718,842443,1952829,8.78,10.5,34.5,-7.7
2024,1548548,558100,63010,135141,506287,1110584,5.58,8.7,32.7,-36.5
2025,1986357,689703,70987,86195,620390,1510344,7.15,4.3,31.2,28.3
2026,902231,399037,47782,0,248643,713635,3.25,0.0,27.6,-54.6


In [3]:
per_bulan = sql('''
    SELECT date_trunc('month', rev_timestamp)::DATE AS bulan,
           count(*) AS jumlah_revisi
    FROM revisi
    GROUP BY 1
    ORDER BY 1
''')

print(f"{len(per_bulan)} bulan: {per_bulan['bulan'][0]} .. {per_bulan['bulan'][-1]}")
print('5 bulan pertama:')
print(per_bulan.head(5))
print('5 bulan terakhir:')
print(per_bulan.tail(5))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2.1s  SELECT date_trunc('month', rev_timestamp)::DATE AS bulan,
279 bulan: 2003-01-01 .. 2026-09-01
5 bulan pertama:
shape: (5, 2)
┌────────────┬───────────────┐
│ bulan      ┆ jumlah_revisi │
│ ---        ┆ ---           │
│ date       ┆ i64           │
╞════════════╪═══════════════╡
│ 2003-01-01 ┆ 1             │
│ 2003-02-01 ┆ 1             │
│ 2003-05-01 ┆ 2             │
│ 2003-10-01 ┆ 6             │
│ 2003-11-01 ┆ 96            │
└────────────┴───────────────┘
5 bulan terakhir:
shape: (5, 2)
┌────────────┬───────────────┐
│ bulan      ┆ jumlah_revisi │
│ ---        ┆ ---           │
│ date       ┆ i64           │
╞════════════╪═══════════════╡
│ 2026-05-01 ┆ 115251        │
│ 2026-06-01 ┆ 102428        │
│ 2026-07-01 ┆ 102795        │
│ 2026-08-01 ┆ 168268        │
│ 2026-09-01 ┆ 3204          │
└────────────┴───────────────┘


In [4]:
hari_jam = sql('''
    SELECT dayofweek(rev_timestamp) AS kode_hari,  -- DuckDB: 0 = Minggu ... 6 = Sabtu
           hour(rev_timestamp) AS jam_utc,         -- 00:00 - 23:00 UTC
           count(*) AS jumlah_revisi
    FROM revisi
    GROUP BY 1, 2
    ORDER BY 1, 2
''')

n_sel, total_hj = hari_jam.height, hari_jam['jumlah_revisi'].sum()
assert n_sel == 7 * 24, f'matriks hari x jam tidak lengkap: {n_sel} sel'
print(f'Matriks hari x jam lengkap: {n_sel} sel, total {total_hj:,} revisi')
hari_jam.head(8)


  1.4s  SELECT dayofweek(rev_timestamp) AS kode_hari,  -- DuckDB: 0 = Ming
Matriks hari x jam lengkap: 168 sel, total 27,771,250 revisi


kode_hari,jam_utc,jumlah_revisi
i64,i64,i64
0,0,139018
0,1,156638
0,2,171253
0,3,180686
0,4,191187
0,5,182656
0,6,186529
0,7,184650


In [5]:
tahun_bulan = sql('''
    SELECT year(rev_timestamp) AS tahun,
           month(rev_timestamp) AS bulan,
           count(*) AS jumlah_revisi
    FROM revisi
    GROUP BY 1, 2
    ORDER BY 1, 2
''')

print(f"{tahun_bulan['tahun'].n_unique()} tahun x {tahun_bulan['bulan'].n_unique()} bulan = {tahun_bulan.height} sel")
tahun_bulan.head(6)


  1.0s  SELECT year(rev_timestamp) AS tahun,
24 tahun x 12 bulan = 279 sel


tahun,bulan,jumlah_revisi
i64,i64,i64
2003,1,1
2003,2,1
2003,5,2
2003,10,6
2003,11,96
2003,12,24


In [6]:
ukuran_artikel = sql('''
    SELECT year(rev_timestamp) AS tahun,
           median(text_bytes) AS median_bytes,
           avg(text_bytes_winsorized) AS rata_rata_bytes,
           count(*) AS jumlah_revisi
    FROM revisi
    WHERE page_namespace = 0   -- hanya artikel utama
    GROUP BY 1
    ORDER BY 1
''')

ukuran_artikel.tail(8)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2.9s  SELECT year(rev_timestamp) AS tahun,


tahun,median_bytes,rata_rata_bytes,jumlah_revisi
i64,f64,f64,i64
2019,1474.0,7778.468696,1360499
2020,4492.0,12176.164586,939033
2021,4681.0,11261.7742,1438735
2022,3899.0,10514.348037,1824173
2023,4237.0,11797.902462,1675693
2024,6220.0,14256.650461,1116351
2025,5900.0,14417.015682,1566643
2026,7347.0,18396.32069,586327


## 2. Agregasi ruang halaman (DuckDB)

Dataset tidak memiliki koordinat geografis, maka **ruang** dipahami sebagai ruang halaman
MediaWiki: distribusi `page_namespace`, halaman paling aktif, dan kontributor teratas.

- **Namespace** = jenis ruang halaman (Artikel, Diskusi, Pengguna, Berkas, Kategori, ...).
- **Halaman** = 20 artikel dengan revisi terbanyak.
- **Kontributor** = 15 penyunting terdaftar (anonim/IP dikelompokkan per alamat, bukan per identitas).


In [7]:
per_namespace = sql('''
    SELECT page_namespace,
           count(*) AS jumlah_revisi,
           count(DISTINCT page_id) AS halaman_unik,
           sum(CASE WHEN is_anonymous THEN 1 ELSE 0 END)::BIGINT AS revisi_anonim
    FROM revisi
    GROUP BY 1
    ORDER BY 2 DESC
''').with_columns(
    (pl.col('jumlah_revisi') / pl.col('jumlah_revisi').sum() * 100).round(2).alias('pct_revisi'),
)

print(f'{len(per_namespace)} namespace terdeteksi')
per_namespace.head(12)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2.4s  SELECT page_namespace,
28 namespace terdeteksi


page_namespace,jumlah_revisi,halaman_unik,revisi_anonim,pct_revisi
i16,i64,i64,i64,f64
0,20358320,1458576,2089690,73.31
3,2181462,1506860,18753,7.86
1,2054374,680658,10452,7.4
14,1061123,202857,11346,3.82
10,650851,110748,22898,2.34
…,…,…,…,…
6,222346,68174,2568,0.8
15,74132,37566,85,0.27
11,45688,16326,295,0.16


In [8]:
# Dua tahap agar count(DISTINCT ...) tidak dihitung pada 1,4 juta grup judul sekaligus
# (pola itu memicu OOM pada memory_limit 1 GB).
top_halaman = sql('''
    WITH top AS (
        SELECT page_title, count(*) AS jumlah_revisi
        FROM revisi
        WHERE page_namespace = 0
        GROUP BY 1
        ORDER BY 2 DESC
        LIMIT 20
    )
    SELECT t.page_title,
           t.jumlah_revisi,
           count(DISTINCT r.contributor_name) AS kontributor_unik,
           min(r.rev_timestamp)::DATE AS revisi_pertama,
           max(r.rev_timestamp)::DATE AS revisi_terakhir,
           sum(CASE WHEN r.is_anonymous THEN 1 ELSE 0 END)::BIGINT AS revisi_anonim
    FROM top AS t
    JOIN revisi AS r
      ON r.page_title = t.page_title AND r.page_namespace = 0
    GROUP BY t.page_title, t.jumlah_revisi
    ORDER BY t.jumlah_revisi DESC
''').with_columns(
    (pl.col('revisi_anonim') / pl.col('jumlah_revisi') * 100).round(1).alias('pct_anonim'),
)

top_halaman


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  3.4s  WITH top AS (


page_title,jumlah_revisi,kontributor_unik,revisi_pertama,revisi_terakhir,revisi_anonim,pct_anonim
str,i64,i64,date,date,i64,f64
"""Daftar acara MDTV""",6672,653,2015-01-11,2026-08-25,1060,15.9
"""Transjakarta""",5285,569,2005-05-10,2026-09-01,653,12.4
"""Daftar stasiun radio di Sulawe…",5172,504,2011-06-20,2026-08-22,3687,71.3
"""D'Academy (musim 7)""",5146,412,2025-06-02,2026-09-01,218,4.2
"""Tanda Nomor Kendaraan Bermotor…",4906,537,2004-02-10,2026-08-31,699,14.2
…,…,…,…,…,…,…
"""Persija Jakarta""",3301,892,2005-05-21,2026-09-01,1069,32.4
"""Tim nasional sepak bola Indone…",3262,834,2004-12-09,2026-08-13,717,22.0
"""Daftar stasiun televisi di Ind…",3256,359,2004-02-10,2026-08-19,204,6.3


In [9]:
# Pola dua tahap yang sama: filter dulu 15 kontributor, baru hitung metrik per kontributor.
top_kontributor = sql('''
    WITH top AS (
        SELECT contributor_name, count(*) AS jumlah_revisi
        FROM revisi
        WHERE NOT is_anonymous AND contributor_name <> 'unknown'
        GROUP BY 1
        ORDER BY 2 DESC
        LIMIT 15
    )
    SELECT t.contributor_name,
           t.jumlah_revisi,
           count(DISTINCT r.page_id) AS halaman_disentuh,
           min(r.rev_timestamp)::DATE AS aktivitas_pertama,
           max(r.rev_timestamp)::DATE AS aktivitas_terakhir,
           sum(CASE WHEN r.is_minor THEN 1 ELSE 0 END)::BIGINT AS revisi_minor,
           round(100.0 * sum(CASE WHEN r.has_comment THEN 1 ELSE 0 END) / count(*), 1) AS pct_berkomentar
    FROM top AS t
    JOIN revisi AS r
      ON r.contributor_name = t.contributor_name AND NOT r.is_anonymous
    GROUP BY t.contributor_name, t.jumlah_revisi
    ORDER BY t.jumlah_revisi DESC
''').with_columns(
    (pl.col('revisi_minor') / pl.col('jumlah_revisi') * 100).round(1).alias('pct_minor'),
)

top_kontributor


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  5.0s  WITH top AS (


contributor_name,jumlah_revisi,halaman_disentuh,aktivitas_pertama,aktivitas_terakhir,revisi_minor,pct_berkomentar,pct_minor
str,i64,i64,date,date,i64,f64,f64
"""Wagino Bot""",4324592,1066155,2011-02-19,2026-02-05,3492153,100.0,80.8
"""HsfBot""",1494678,953359,2016-10-04,2026-09-01,944685,99.9,63.2
"""AABot""",971206,536480,2010-05-21,2026-08-31,956670,99.7,98.5
"""InternetArchiveBot""",825396,566562,2021-01-16,2026-09-01,27,100.0,0.0
"""Kenrick95Bot""",653448,620988,2009-12-12,2020-09-18,118153,100.0,18.1
…,…,…,…,…,…,…,…
"""Arya-Bot""",258469,252954,2011-02-13,2023-03-28,257254,100.0,99.5
"""Bennylin""",257365,152367,2004-12-05,2026-08-12,116867,68.5,45.4
"""Borgxbot""",252387,177633,2006-01-04,2009-10-30,193054,99.3,76.5


## 3. Transformasi turunan (Polars)

Hasil agregat di atas (ratusan baris) ditransformasi dengan Polars -- bukan DuckDB maupun pandas --
agar alur kerja tetap ringan:

- **Rata-rata bergerak 12 bulan** dan perubahan bulanan (`mom_pct`) pada deret bulanan.
- **Pivot** dua matriks: tahun x bulan dan hari x jam (untuk heatmap).
- **Rasio** penyunting anonim per tahun (turunan dari agregat, bukan hitung ulang data mentah).


In [10]:
df_bulan = (
    per_bulan.sort('bulan')
    .with_columns(
        pl.col('jumlah_revisi')
        .rolling_mean(window_size=12)
        .round(0)
        .alias('rata_rata_12_bulan'),
        ((pl.col('jumlah_revisi') / pl.col('jumlah_revisi').shift(1) - 1) * 100)
        .round(1)
        .alias('mom_pct'),
    )
)

puncak_bulan = df_bulan.sort('jumlah_revisi', descending=True).head(6)
print('Enam bulan dengan revisi terbanyak:')
puncak_bulan


Enam bulan dengan revisi terbanyak:


bulan,jumlah_revisi,rata_rata_12_bulan,mom_pct
date,i64,f64,f64
2022-01-01,796415,200914.0,520.0
2016-02-01,601498,203214.0,185.5
2019-10-01,398278,140985.0,104.8
2025-11-01,373007,166648.0,4.5
2025-10-01,357031,143921.0,162.7
2023-10-01,327273,218330.0,24.5


In [11]:
matriks_tahun_bulan = tahun_bulan.pivot(
    values='jumlah_revisi', index='tahun', on='bulan', aggregate_function='sum'
)
matriks_hari_jam = hari_jam.pivot(
    values='jumlah_revisi', index='kode_hari', on='jam_utc', aggregate_function='sum'
)

# Urutkan kolom (bulan/jam) dan baris secara numerik: dengan preserve_insertion_order=false,
# urutan hasil pivot tidak terjamin -- jika dibiarkan, label heatmap bisa salah pasang.
kolom_bulan = sorted((c for c in matriks_tahun_bulan.columns if c != 'tahun'), key=int)
matriks_tahun_bulan = matriks_tahun_bulan.select(['tahun'] + kolom_bulan).sort('tahun')
kolom_jam = sorted((c for c in matriks_hari_jam.columns if c != 'kode_hari'), key=int)
matriks_hari_jam = matriks_hari_jam.select(['kode_hari'] + kolom_jam).sort('kode_hari')

print('matriks tahun x bulan:', matriks_tahun_bulan.shape, '| kolom bulan:', kolom_bulan)
print('matriks hari x jam   :', matriks_hari_jam.shape, '| kolom jam  :', kolom_jam)
assert [int(c) for c in kolom_bulan] == list(range(1, 13)), 'kolom bulan tidak lengkap/tidak urut'
assert [int(c) for c in kolom_jam] == list(range(24)), 'kolom jam tidak lengkap/tidak urut'
matriks_tahun_bulan.head(5)


matriks tahun x bulan: (24, 13) | kolom bulan: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12']
matriks hari x jam   : (7, 25) | kolom jam  : ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23']


tahun,1,2,3,4,5,6,7,8,9,10,11,12
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2003,1,1,0,0,2,0,0,0,0,6,96,24
2004,121,708,1462,1059,1132,523,1081,755,394,1291,2634,4015
2005,3269,2370,4358,4942,6465,9912,11175,9943,10442,10082,9162,12105
2006,17078,16812,30470,26710,21430,23285,27513,31797,26947,40626,38601,32244
2007,48377,37838,44175,78450,61187,37028,36709,39244,47262,51033,50849,48233


## 4. Visualisasi interaktif (Plotly)

Delapan figur interaktif (memenuhi syarat minimal 6 visualisasi interaktif Milestone 3):

| # | Figur | Pertanyaan |
|---|---|---|
| 1 | Revisi per tahun (stack anonim/terdaftar + garis total) | Tren & penyunting |
| 2 | Revisi per bulan + rata-rata 12 bulan + *rangeslider* | Musiman jangka panjang |
| 3 | Heatmap hari x jam UTC | Pola aktivitas harian |
| 4 | Heatmap tahun x bulan | Kalender musiman & lonjakan |
| 5 | Sunburst namespace | Ruang halaman |
| 6 | 20 halaman teratas (stack anonim/terdaftar) | Konsentrasi topik |
| 7 | 15 kontributor teratas (warna = % berkomentar) | Konsentrasi penyunting |
| 8 | Median & rata-rata ukuran teks per tahun (log) | Pertumbuhan konten |

Setiap figur disimpan sebagai HTML interaktif di `output/figures/` pada akhir notebook.


In [12]:
fig1_df = (
    per_tahun
    .with_columns((pl.col('jumlah_revisi') - pl.col('revisi_anonim')).alias('revisi_terdaftar'))
    .select('tahun', 'revisi_anonim', 'revisi_terdaftar')
    .unpivot(
        on=['revisi_anonim', 'revisi_terdaftar'],
        index='tahun',
        variable_name='jenis',
        value_name='jumlah_revisi',
    )
    .to_pandas()
    .replace({'revisi_anonim': 'Anonim (IP)', 'revisi_terdaftar': 'Terdaftar'})
)

fig1 = px.bar(
    fig1_df,
    x='tahun',
    y='jumlah_revisi',
    color='jenis',
    barmode='stack',
    title='Revisi Wikipedia Indonesia per tahun (2003-2026)',
    labels={'jumlah_revisi': 'Jumlah revisi', 'jenis': 'Penyunting', 'tahun': 'Tahun'},
    color_discrete_map={'Anonim (IP)': '#d62728', 'Terdaftar': '#1f77b4'},
    hover_data={'tahun': True},
)
fig1.add_trace(
    go.Scatter(
        x=per_tahun['tahun'].to_numpy(),
        y=per_tahun['jumlah_revisi'].to_numpy(),
        mode='lines+markers',
        name='Total revisi',
        line=dict(color='#2ca02c', width=2),
        hovertemplate='Tahun %{x}<br>Total %{y:,.0f} revisi<extra></extra>',
    )
)
fig1.update_layout(legend_title_text='Penyunting', hovermode='x unified')
fig1.show()


In [13]:
fig2_df = df_bulan.to_pandas().rename(columns={
    'jumlah_revisi': 'Revisi bulanan',
    'rata_rata_12_bulan': 'Rata-rata 12 bulan',
})

fig2 = px.line(
    fig2_df,
    x='bulan',
    y=['Revisi bulanan', 'Rata-rata 12 bulan'],
    title='Revisi per bulan (2003-2026) dan rata-rata bergerak 12 bulan',
    labels={'value': 'Jumlah revisi', 'variable': 'Serie', 'bulan': 'Bulan'},
)
fig2.update_xaxes(rangeslider_visible=True, rangeselector=dict(
    buttons=[
        dict(count=1, label='1 tahun', step='year', stepmode='backward'),
        dict(count=3, label='3 tahun', step='year', stepmode='backward'),
        dict(count=10, label='10 tahun', step='year', stepmode='backward'),
        dict(step='all', label='Semua'),
    ]
))
fig2.update_layout(hovermode='x unified')
fig2.show()


In [14]:
NAMA_HARI = {0: 'Minggu', 1: 'Senin', 2: 'Selasa', 3: 'Rabu', 4: 'Kamis', 5: 'Jumat', 6: 'Sabtu'}
URUTAN_HARI = [1, 2, 3, 4, 5, 6, 0]  # Senin .. Minggu

mat_hari_jam = matriks_hari_jam.to_pandas().set_index('kode_hari').loc[URUTAN_HARI]
mat_hari_jam.index = [NAMA_HARI[h] for h in URUTAN_HARI]

fig3 = px.imshow(
    mat_hari_jam.values,
    x=[f'{h:02d}' for h in range(24)],
    y=list(mat_hari_jam.index),
    labels=dict(x='Jam (UTC)', y='Hari', color='Jumlah revisi'),
    text_auto='.2s',
    aspect='auto',
    color_continuous_scale='YlOrRd',
    title='Intensitas revisi: hari dalam minggu x jam UTC (2003-2026)',
)
fig3.add_annotation(
    text='Jam 13-14 UTC = 20:00-21:00 WIB (UTC+7)',
    xref='paper', yref='paper', x=0, y=-0.22, showarrow=False, font=dict(size=12),
)
fig3.update_yaxes(autorange='reversed')
fig3.show()


In [15]:
mat_tahun_bulan = matriks_tahun_bulan.to_pandas().set_index('tahun')
urutan_tahun = list(mat_tahun_bulan.index[::-1])  # tahun terbaru di atas

fig4 = px.imshow(
    mat_tahun_bulan.values[::-1],
    x=[str(m) for m in mat_tahun_bulan.columns],
    y=[str(y) for y in urutan_tahun],
    labels=dict(x='Bulan', y='Tahun', color='Jumlah revisi'),
    text_auto='.2s',
    aspect='auto',
    color_continuous_scale='Viridis',
    title='Kalender revisi: tahun x bulan (musiman dan lonjakan)',
)
fig4.show()


In [16]:
NAMA_NS = {
    0: 'Artikel', 1: 'Diskusi artikel', 2: 'Diskusi pengguna', 3: 'Pengguna',
    4: 'Wikipedia', 5: 'Diskusi Wikipedia', 6: 'Berkas', 7: 'Diskusi berkas',
    8: 'MediaWiki', 9: 'Diskusi MediaWiki', 10: 'Template', 11: 'Diskusi template',
    12: 'Bantuan', 13: 'Diskusi bantuan', 14: 'Kategori', 15: 'Diskusi kategori',
}
KELOMPOK_NS = {
    0: 'Artikel', 1: 'Artikel', 2: 'Pengguna', 3: 'Pengguna',
    4: 'Wikipedia', 5: 'Wikipedia', 6: 'Berkas', 7: 'Berkas',
    8: 'MediaWiki', 9: 'MediaWiki', 10: 'Template', 11: 'Template',
    12: 'Bantuan', 13: 'Bantuan', 14: 'Kategori', 15: 'Kategori',
}

ns_pd = per_namespace.to_pandas()
ns_pd['namespace'] = ns_pd['page_namespace'].map(
    lambda n: NAMA_NS.get(int(n), f'ns {int(n)} non-standar')
)
ns_pd['kelompok'] = ns_pd['page_namespace'].map(
    lambda n: KELOMPOK_NS.get(int(n), 'Non-standar')
)

fig5 = px.sunburst(
    ns_pd,
    path=['kelompok', 'namespace'],
    values='jumlah_revisi',
    color='kelompok',
    hover_data={'pct_revisi': True, 'halaman_unik': True, 'revisi_anonim': True},
    title='Distribusi revisi menurut namespace halaman (ruang halaman)',
)
fig5.update_layout(margin=dict(t=40, l=0, r=0, b=0))
fig5.show()


In [17]:
fig6_df = (
    top_halaman
    .with_columns((pl.col('jumlah_revisi') - pl.col('revisi_anonim')).alias('revisi_terdaftar'))
    .select('page_title', 'revisi_anonim', 'revisi_terdaftar')
    .unpivot(
        on=['revisi_anonim', 'revisi_terdaftar'],
        index='page_title',
        variable_name='jenis',
        value_name='jumlah_revisi',
    )
    .to_pandas()
    .replace({'revisi_anonim': 'Anonim (IP)', 'revisi_terdaftar': 'Terdaftar'})
)

fig6 = px.bar(
    fig6_df,
    y='page_title',
    x='jumlah_revisi',
    color='jenis',
    orientation='h',
    barmode='stack',
    title='20 artikel paling banyak direvisi (2003-2026)',
    labels={'page_title': 'Halaman (namespace 0)', 'jumlah_revisi': 'Jumlah revisi', 'jenis': 'Penyunting'},
    color_discrete_map={'Anonim (IP)': '#d62728', 'Terdaftar': '#1f77b4'},
)
fig6.update_yaxes(autorange='reversed')
fig6.update_layout(legend_title_text='Penyunting')
fig6.show()


In [18]:
fig7 = px.bar(
    top_kontributor.to_pandas(),
    y='contributor_name',
    x='jumlah_revisi',
    orientation='h',
    color='pct_berkomentar',
    color_continuous_scale='Blues',
    title='15 kontributor terdaftar dengan revisi terbanyak',
    labels={
        'contributor_name': 'Kontributor',
        'jumlah_revisi': 'Jumlah revisi',
        'pct_berkomentar': '% revisi berkomentar',
    },
    hover_data={
        'halaman_disentuh': True,
        'pct_minor': True,
        'aktivitas_pertama': True,
        'aktivitas_terakhir': True,
        'jumlah_revisi': ':,',
    },
)
fig7.update_yaxes(autorange='reversed')
fig7.update_layout(coloraxis_colorbar_title='% berkomentar')
fig7.show()


In [19]:
fig8_df = ukuran_artikel.to_pandas().rename(columns={
    'median_bytes': 'Median (byte)',
    'rata_rata_bytes': 'Rata-rata winsor (byte)',
})

fig8 = px.line(
    fig8_df,
    x='tahun',
    y=['Median (byte)', 'Rata-rata winsor (byte)'],
    log_y=True,
    markers=True,
    title='Ukuran teks artikel (namespace 0) per tahun -- skala logaritmik',
    labels={'value': 'Ukuran teks (byte)', 'variable': 'Metrik', 'tahun': 'Tahun'},
)
fig8.update_layout(hovermode='x unified')
fig8.show()


## 5. Verifikasi angka insight

Setiap angka yang dikutip pada Bagian 6 dihitung ulang di sini langsung dari 27.771.250 baris,
sehingga seluruh insight dapat ditelusuri ke bukti eksekusi notebook -- bukan angka manual.


In [20]:
# (a) Rasio per periode -> dasar insight I7
periode = sql('''
    SELECT CASE
             WHEN rev_timestamp < TIMESTAMP '2010-01-01' THEN '2003-2009'
             WHEN rev_timestamp < TIMESTAMP '2016-01-01' THEN '2010-2015'
             WHEN rev_timestamp < TIMESTAMP '2022-01-01' THEN '2016-2021'
             ELSE '2022-2026'
           END AS periode,
           count(*) AS jumlah_revisi,
           round(100.0 * sum(CASE WHEN is_anonymous THEN 1 ELSE 0 END) / count(*), 1) AS pct_anonim,
           round(100.0 * sum(CASE WHEN is_minor THEN 1 ELSE 0 END) / count(*), 1) AS pct_minor,
           round(100.0 * sum(CASE WHEN has_comment THEN 1 ELSE 0 END) / count(*), 1) AS pct_berkomentar
    FROM revisi
    GROUP BY 1
    ORDER BY 1
''')
print('(a) Rasio per periode -> insight I7')
print(periode)

# (b) Konsentrasi revisi pada halaman -> dasar insight I6
konsentrasi = sql('''
    WITH s AS (
        SELECT page_title, count(*) AS n FROM revisi WHERE page_namespace = 0 GROUP BY 1
    ),
    r AS (
        SELECT n, row_number() OVER (ORDER BY n DESC) AS rk, sum(n) OVER () AS tot FROM s
    )
    SELECT (SELECT count(*) FROM s) AS halaman_artikel,
           sum(n) FILTER (WHERE rk <= 0.01 * (SELECT count(*) FROM s)) AS revisi_top_1pct,
           round(100.0 * sum(n) FILTER (WHERE rk <= 0.01 * (SELECT count(*) FROM s))
                 / max(tot), 1) AS pct_top_1pct
    FROM r
''')
print('(b) Konsentrasi revisi halaman -> insight I6')
print(konsentrasi)

# (c) Kontributor teratas -> dasar insight I1/I3
konsentrasi_kontributor = sql('''
    WITH top AS (
        SELECT contributor_name, count(*) AS n
        FROM revisi
        WHERE NOT is_anonymous AND contributor_name <> 'unknown'
        GROUP BY 1
        ORDER BY 2 DESC
        LIMIT 12
    )
    SELECT (SELECT count(DISTINCT contributor_name) FROM revisi) AS nama_unik,
           (SELECT count(DISTINCT CASE WHEN NOT is_anonymous AND contributor_name <> 'unknown'
                      THEN contributor_name END) FROM revisi) AS kontributor_terdaftar_unik,
           sum(n) AS revisi_top_12,
           round(100.0 * sum(n) / (SELECT count(*) FROM revisi), 1) AS pct_top_12
    FROM top
''')
print('(c) Kontributor teratas -> insight I3')
print(konsentrasi_kontributor)

# (d) Komposisi lonjakan Januari 2022 -> dasar insight I2
lonjakan = sql('''
    SELECT count(*) AS revisi_jan_2022,
           round(100.0 * sum(CASE WHEN is_anonymous THEN 1 ELSE 0 END) / count(*), 1) AS pct_anonim,
           round(100.0 * sum(CASE WHEN is_minor THEN 1 ELSE 0 END) / count(*), 1) AS pct_minor,
           sum(CASE WHEN contributor_name = 'Wagino Bot' THEN 1 ELSE 0 END) AS revisi_wagino_bot,
           round(100.0 * sum(CASE WHEN contributor_name = 'Wagino Bot' THEN 1 ELSE 0 END)
                 / count(*), 1) AS pct_wagino_bot
    FROM revisi
    WHERE rev_timestamp >= TIMESTAMP '2022-01-01'
      AND rev_timestamp < TIMESTAMP '2022-02-01'
''')
print('(d) Lonjakan Januari 2022 -> insight I2')
print(lonjakan)


  2.0s  SELECT CASE
(a) Rasio per periode -> insight I7
shape: (4, 5)
┌───────────┬───────────────┬────────────┬───────────┬─────────────────┐
│ periode   ┆ jumlah_revisi ┆ pct_anonim ┆ pct_minor ┆ pct_berkomentar │
│ ---       ┆ ---           ┆ ---        ┆ ---       ┆ ---             │
│ str       ┆ i64           ┆ f64        ┆ f64       ┆ f64             │
╞═══════════╪═══════════════╪════════════╪═══════════╪═════════════════╡
│ 2003-2009 ┆ 2530858       ┆ 9.3        ┆ 65.2      ┆ 83.0            │
│ 2010-2015 ┆ 7165105       ┆ 7.9        ┆ 53.4      ┆ 83.8            │
│ 2016-2021 ┆ 8557945       ┆ 7.9        ┆ 46.6      ┆ 81.8            │
│ 2022-2026 ┆ 9517342       ┆ 7.6        ┆ 35.0      ┆ 77.9            │
└───────────┴───────────────┴────────────┴───────────┴─────────────────┘


  1.5s  WITH s AS (
(b) Konsentrasi revisi halaman -> insight I6
shape: (1, 3)
┌─────────────────┬─────────────────┬──────────────┐
│ halaman_artikel ┆ revisi_top_1pct ┆ pct_top_1pct │
│ ---             ┆ ---             ┆ ---          │
│ i64             ┆ decimal[38,0]   ┆ f64          │
╞═════════════════╪═════════════════╪══════════════╡
│ 1458576         ┆ 4821575         ┆ 23.7         │
└─────────────────┴─────────────────┴──────────────┘


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2.8s  WITH top AS (
(c) Kontributor teratas -> insight I3
shape: (1, 4)
┌───────────┬────────────────────────────┬───────────────┬────────────┐
│ nama_unik ┆ kontributor_terdaftar_unik ┆ revisi_top_12 ┆ pct_top_12 │
│ ---       ┆ ---                        ┆ ---           ┆ ---        │
│ i64       ┆ i64                        ┆ decimal[38,0] ┆ f64        │
╞═══════════╪════════════════════════════╪═══════════════╪════════════╡
│ 966299    ┆ 331071                     ┆ 10516615      ┆ 37.9       │
└───────────┴────────────────────────────┴───────────────┴────────────┘


  1.7s  SELECT count(*) AS revisi_jan_2022,
(d) Lonjakan Januari 2022 -> insight I2
shape: (1, 5)
┌─────────────────┬────────────┬───────────┬───────────────────┬────────────────┐
│ revisi_jan_2022 ┆ pct_anonim ┆ pct_minor ┆ revisi_wagino_bot ┆ pct_wagino_bot │
│ ---             ┆ ---        ┆ ---       ┆ ---               ┆ ---            │
│ i64             ┆ f64        ┆ f64       ┆ decimal[38,0]     ┆ f64            │
╞═════════════════╪════════════╪═══════════╪═══════════════════╪════════════════╡
│ 796415          ┆ 2.5        ┆ 70.2      ┆ 573119            ┆ 72.0           │
└─────────────────┴────────────┴───────────┴───────────────────┴────────────────┘


In [21]:
# (e) Ringkasan pola waktu dari matriks hari x jam (transformasi Polars) -> insight I4
total_seluruh = hari_jam['jumlah_revisi'].sum()

per_hari = (
    hari_jam.group_by('kode_hari')
    .agg(pl.col('jumlah_revisi').sum())
    .sort('kode_hari')
    .with_columns(
        pl.col('kode_hari').cast(pl.Int64).replace_strict(NAMA_HARI, default='?').alias('hari'),
        (pl.col('jumlah_revisi') / total_seluruh * 100).round(1).alias('pct_revisi'),
    )
    .select('hari', 'jumlah_revisi', 'pct_revisi')
)
print('(e) Total revisi per hari -> insight I4')
print(per_hari)

per_jam = (
    hari_jam.group_by('jam_utc')
    .agg(pl.col('jumlah_revisi').sum())
    .sort('jam_utc')
    .with_columns((pl.col('jumlah_revisi') / total_seluruh * 100).round(2).alias('pct_revisi'))
)
jam_puncak = per_jam.sort('jumlah_revisi', descending=True).head(3)
n_siang = per_jam.filter(pl.col('jam_utc').is_between(12, 16))['jumlah_revisi'].sum()
n_malam = per_jam.filter((pl.col('jam_utc') >= 20) | (pl.col('jam_utc') <= 3))['jumlah_revisi'].sum()
print('(e) Tiga jam puncak (UTC) -> insight I4')
print(jam_puncak)
print(f'jam 12-16 UTC : {n_siang / total_seluruh * 100:.1f}% revisi')
print(f'jam 20-03 UTC : {n_malam / total_seluruh * 100:.1f}% revisi')


(e) Total revisi per hari -> insight I4
shape: (7, 3)
┌────────┬───────────────┬────────────┐
│ hari   ┆ jumlah_revisi ┆ pct_revisi │
│ ---    ┆ ---           ┆ ---        │
│ str    ┆ i64           ┆ f64        │
╞════════╪═══════════════╪════════════╡
│ Minggu ┆ 3765633       ┆ 13.6       │
│ Senin  ┆ 3835389       ┆ 13.8       │
│ Selasa ┆ 3994920       ┆ 14.4       │
│ Rabu   ┆ 3949023       ┆ 14.2       │
│ Kamis  ┆ 4184755       ┆ 15.1       │
│ Jumat  ┆ 3951044       ┆ 14.2       │
│ Sabtu  ┆ 4090486       ┆ 14.7       │
└────────┴───────────────┴────────────┘
(e) Tiga jam puncak (UTC) -> insight I4
shape: (3, 3)
┌─────────┬───────────────┬────────────┐
│ jam_utc ┆ jumlah_revisi ┆ pct_revisi │
│ ---     ┆ ---           ┆ ---        │
│ i64     ┆ i64           ┆ f64        │
╞═════════╪═══════════════╪════════════╡
│ 14      ┆ 1463102       ┆ 5.27       │
│ 13      ┆ 1462276       ┆ 5.27       │
│ 9       ┆ 1449788       ┆ 5.22       │
└─────────┴───────────────┴────────────┘
jam

## 6. Insight analitik

Seluruh angka di bawah berasal dari eksekusi notebook ini (Bagian 1-5).

**I1 -- Skala dan tren pertumbuhan melambat di akhir periode.**
Dataset memuat **27.771.250 revisi** pada **4.253.661 halaman** dari **966.299 nilai
`contributor_name` unik** (966.298 kontributor + label `unknown`) pada rentang
**24 Jan 2003 - 1 Sep 2026**. Volume tahunan memuncak pada **2022 (2.641.295 revisi)**, disusul
2023 (2.438.911); setelah itu berkisar 1,5-2,0 juta revisi/tahun (2024: 1.548.548;
2025: 1.986.357). Sepanjang 2026 (8 bulan) tercatat
**902.231 revisi** -- setara laju ~1,35 juta/tahun, jauh di bawah puncak 2022. → *Figur 1, 2*.

**I2 -- Lonjakan Januari 2022 hampir seluruhnya berasal dari satu bot.**
**Januari 2022 menyumbang 796.415 revisi (2,9% total seluruh data) dalam satu bulan**, dengan
hanya **2,5% revisi anonim** dan **70,2% berflag minor** -- profil khas otomasi, bukan gerakan
penyunting manusia. Penyumbang terbesarnya adalah **Wagino Bot: 573.119 revisi (72% bulan itu)**.
Pola serupa terlihat pada **Februari 2016 (601.498 revisi; anonim hanya 1,2%)**. → *Figur 2, 4*.

**I3 -- Aktivitas penyunting sangat terkonsentrasi pada sedikit akun bot.**
Lima kontributor teratas **semuanya bot**; **Wagino Bot sendiri menyumbang 4.324.592 revisi = 15,6%
seluruh dataset** dengan 1.066.155 halaman disentuh. Gabungan 12 kontributor teratas =
**10.516.615 revisi (37,9%)**, sementara hanya ada **331.071 kontributor terdaftar unik** dan
**7,9% revisi (2.193.565) berasal dari penyunting anonim/IP**. → *Figur 7*.

**I4 -- Pola waktu lebih kuat pada skala jam daripada skala hari.**
Jam puncak adalah **13:00-14:00 UTC = 20:00-21:00 WIB** (1.463.102 dan 1.462.276 revisi);
**24,8% revisi** terjadi pada jam 12-16 UTC dan **26,4%** pada jam 20-03 UTC -- aktivitas
menumpuk pada malam hari WIB. Sebaliknya perbedaan hari kecil: **Kamis tertinggi 4.184.755
(15,1%)** vs **Minggu terendah 3.765.633 (13,6%)**, hanya selisih ~1,5 poin persen. → *Figur 3*.

**I5 -- Ruang halaman tetap terpusat pada artikel, tapi lapisan non-artikel besar.**
**Namespace Artikel (0) = 73,31% revisi (20.358.320)**; ruang **Pengguna (3) 7,86%** bahkan
melampaui **Diskusi artikel (1) 7,40%**, lalu **Kategori (14) 3,82%** dan **Template (10) 2,34%**.
Terdapat **28 namespace** berbeda termasuk ruang non-standar (2600, 1704, 828) -- ruang kerja
internal ikut aktif menyunting. → *Figur 5*.

**I6 -- Revisi menumpuk pada artikel 'daftar' dan halaman layanan publik.**
**1% halaman artikel teratas (14.586 halaman) menanggung 23,7% seluruh revisi artikel**.
Peringkat teratas didominasi pola *listicle* dan layanan publik: **Daftar acara MDTV (6.672)**,
**Transjakarta (5.285)**, **Daftar stasiun radio di Sulawesi Utara (5.172)**, D'Academy musim 7
(5.146), Tanda Nomor Kendaraan Bermotor (4.906). Rasio anonim sangat bervariasi antar halaman:
**71,3%** untuk daftar stasiun radio Sulawesi Utara vs **4,2%** untuk D'Academy. → *Figur 6*.

**I7 -- Suntingan makin besar dan makin sedikit yang berflag minor.**
Rasio **revisi minor anjlok dari 65,2% (2003-2009) menjadi 35,0% (2022-2026)**, sementara
revisi berketerangan turun dari 83,0% ke 77,9% dan anonim dari 9,3% ke 7,6%. Perubahan ini sejalan
dengan dominasi bot berskala besar (I3) dan makin banyaknya suntingan substansial. → *Figur 1*.

**I8 -- Ukuran artikel tumbuh hampir 9 kali lipat.**
Median `text_bytes` artikel naik dari **814 byte (2004)** ke **2.455 (2010)**, **3.899 (2022)**,
**5.900 (2025)**, dan **7.347 byte (2026)**; rata-rata winsor ikut naik dari 2.286 ke 18.396 byte.
Konten Wikipedia Indonesia bertambah panjang seiring waktu. → *Figur 8*.


In [22]:
FIGURE = {
    '01_revisi_per_tahun': fig1,
    '02_revisi_per_bulan': fig2,
    '03_heatmap_hari_jam_utc': fig3,
    '04_heatmap_tahun_bulan': fig4,
    '05_distribusi_namespace': fig5,
    '06_top_halaman': fig6,
    '07_top_kontributor': fig7,
    '08_ukuran_teks_per_tahun': fig8,
}

assert len(FIGURE) >= 6, 'Milestone 3 mensyaratkan minimal 6 visualisasi interaktif'

for nama, figur in FIGURE.items():
    tujuan = FIG_DIR / f'{nama}.html'
    figur.write_html(tujuan, include_plotlyjs='cdn')
    print(f'{tujuan}  ({tujuan.stat().st_size / 1024:.0f} KB)')

print(f'\n{len(FIGURE)} figur interaktif diekspor ke {FIG_DIR.resolve()}')


../output/figures/01_revisi_per_tahun.html  (9 KB)
../output/figures/02_revisi_per_bulan.html  (29 KB)
../output/figures/03_heatmap_hari_jam_utc.html  (9 KB)


../output/figures/04_heatmap_tahun_bulan.html  (10 KB)


../output/figures/05_distribusi_namespace.html  (12 KB)
../output/figures/06_top_halaman.html  (10 KB)


../output/figures/07_top_kontributor.html  (10 KB)
../output/figures/08_ukuran_teks_per_tahun.html  (9 KB)

8 figur interaktif diekspor ke /home/jovyan/work/output/figures


## Catatan Milestone 3

1. **Deliverable terpenuhi**: analisis temporal (tahun, bulan, hari, jam UTC) dan analisis ruang
   (namespace, halaman, kontributor) dijalankan penuh pada 27.771.250 baris; **8 visualisasi
   interaktif Plotly** (syarat: minimal 6) ditampilkan di notebook dan diekspor sebagai HTML ke
   `output/figures/` (8 insight analitik berangka tercantum di Bagian 6, syarat final: minimal 5).
2. **Pembagian peran sesuai standar kuliah**: DuckDB -- *scan*/agregasi berat (view `revisi`,
   `memory_limit` 1 GB, spill `/tmp`); Polars -- transformasi turunan (rolling mean, pivot matriks,
   rasio, YoY) dalam mode lazy pada hasil agregat kecil. Tidak ada `collect` data mentah ke RAM.
3. **Semua angka direproduksi dari data** saat notebook dieksekusi; tidak ada angka hard-coded.
   Total baris yang dipakai konsisten dengan hasil Milestone 2 (27.771.250 baris, 0 baris dihapus).
4. **Temuan utama**: aktivitas terkonsentrasi pada bot (1 akun = 15,6% revisi), lonjakan Jan 2022
   didominasi satu bot, pola aktivitas menumpuk pada malam WIB, dan ukuran artikel tumbuh ~9x.
5. **Keterbatasan**: timestamp dalam UTC tanpa zona penyunting; penyunting anonim diidentifikasi
   per alamat IP (bukan per orang); tahun 2026 hanya berjalan sampai 1 September sehingga tidak
   sebanding penuh dengan tahun lain.
